# Zurich Transit Subset Derivation

This notebook documents the exploratory analysis used to derive the Zurich operational subset Swiss GTFS-S 2026 feed.

The objective is to reduce the nationwide timetable dataset into a transit network centered on Zurich while preserving operationally relevant services

The notebook is a research artifact only.

The production implementation exists separately in:

data/scripts/transit_subset

In [2]:
from pathlib import Path

import polars as pl

In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "raw"

GTFS_DIR = sorted(
    RAW_DIR.glob("gtfs_fp*"),
    reverse=True
)[0]

GTFS_DIR

PosixPath('/mnt/d/transit-intelligence/packages/gtfs_s/raw/gtfs_fp2026_20260617')

## Dataset Inventory

First, determine the size of each GTFS table.

This establishes the scale of the nationwide dataset before any filtering.

In [5]:
PARQUET_DIR = PROJECT_ROOT / "parquet"

In [6]:
summary = []

PARQUET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for file in sorted(GTFS_DIR.glob("*.txt")):

    print(f"Processing {file.name}...")

    scan = pl.scan_csv(file)

    rows = (
        scan
        .select(pl.len())
        .collect()
        .item()
    )

    schema = scan.collect_schema()

    parquet_file = (
        PARQUET_DIR
        / f"{file.stem}.parquet"
    )

    if not parquet_file.exists():

        print(
            f"  Converting -> {parquet_file.name}"
        )

        (
            scan
            .collect(streaming=True)
            .write_parquet(parquet_file)
        )

    else:

        print(
            f"  Already exists -> {parquet_file.name}"
        )

    summary.append(
        {
            "table": file.stem,
            "rows": rows,
            "columns": len(schema),
            "parquet": parquet_file.name,
        }
    )

summary_df = (
    pl.DataFrame(summary)
    .sort("rows", descending=True)
)

summary_df

Processing agency.txt...
  Converting -> agency.parquet


/tmp/ipykernel_9185/2619100903.py:36: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


Processing calendar.txt...
  Converting -> calendar.parquet
Processing calendar_dates.txt...
  Converting -> calendar_dates.parquet
Processing feed_info.txt...
  Converting -> feed_info.parquet
Processing frequencies.txt...
  Converting -> frequencies.parquet
Processing routes.txt...
  Converting -> routes.parquet
Processing stop_times.txt...
  Converting -> stop_times.parquet
Processing stops.txt...
  Converting -> stops.parquet
Processing transfers.txt...
  Converting -> transfers.parquet
Processing trips.txt...
  Converting -> trips.parquet


table,rows,columns,parquet
str,i64,i64,str
"""stop_times""",25132289,7,"""stop_times.parquet"""
"""calendar_dates""",8867454,3,"""calendar_dates.parquet"""
"""trips""",1584909,9,"""trips.parquet"""
"""transfers""",255478,8,"""transfers.parquet"""
"""stops""",102796,9,"""stops.parquet"""
"""calendar""",58791,10,"""calendar.parquet"""
"""routes""",5076,6,"""routes.parquet"""
"""frequencies""",1868,5,"""frequencies.parquet"""
"""agency""",475,6,"""agency.parquet"""


## Stop Inventory

The Swiss GTFS feed contains over 100,000 stop records.

The first task is determining which stops belong to the Zurich operational area.

In [7]:
stops = pl.read_parquet(
    PARQUET_DIR / "stops.parquet"
)

stops.head()

stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station,platform_code,original_stop_id,didok
str,str,f64,f64,str,str,str,str,i64
"""7104307""","""Figueras Vilafant""",42.264779,2.9430246,"""""","""Parent7104307""","""""","""7104307""",7104307
"""7171801""","""Barcelona Sants""",41.378914,2.140371,"""""","""Parent7171801""","""""","""7171801""",7171801
"""7179300""","""Gerona""",41.979519,2.816488,"""""","""Parent7179300""","""""","""7179300""",7179300
"""8002140""","""Augsburg Hbf""",48.365441,10.885569,"""""","""Parent8002140""","""""","""8002140""",8002140
"""8002301""","""Lindau-Reutin""",47.552384,9.703296,"""""","""Parent8002301""","""""","""8002301""",8002301


## Radius-Based Exploration

Initial exploration used Zürich HB as the reference point.

In [8]:
import math

ZURICH_HB_LAT = 47.378177
ZURICH_HB_LON = 8.540192

EARTH_RADIUS_KM = 6371.0

stops = stops.with_columns(
    (
        2
        * EARTH_RADIUS_KM
        * (
            (
                (
                    (
                        (pl.col("stop_lat").radians() - math.radians(ZURICH_HB_LAT))
                        / 2
                    )
                    .sin()
                    .pow(2)
                )
                + (
                    math.cos(math.radians(ZURICH_HB_LAT))
                    * pl.col("stop_lat").radians().cos()
                    * (
                        (
                            pl.col("stop_lon").radians()
                            - math.radians(ZURICH_HB_LON)
                        )
                        / 2
                    )
                    .sin()
                    .pow(2)
                )
            )
            .sqrt()
            .arcsin()
        )
    ).alias("distance_km")
)

In [9]:
radius_summary = []

for radius in [5, 10, 15, 20, 25]:

    count = (
        stops
        .filter(
            pl.col("distance_km") <= radius
        )
        .height
    )

    radius_summary.append(
        {
            "radius_km": radius,
            "stops": count,
        }
    )

pl.DataFrame(radius_summary)

radius_km,stops
i64,i64
5,1852
10,4070
15,6098
20,8486
25,11597


### Observation

Distance filtering successfully approximates the Zurich metropolitan area.

However, operational boundaries remain ambiguous.

A naming-based exploration was performed next.

In [10]:
zurich_stops = stops.filter(
    pl.col("stop_name")
    .str.starts_with("Zürich")
)

print(
    f"Rows: {zurich_stops.height:,}"
)

print(
    f"Unique names: "
    f"{zurich_stops.select(pl.col('stop_name').n_unique()).item():,}"
)

Rows: 2,007
Unique names: 472


In [11]:
(
    zurich_stops
    .group_by("stop_name")
    .len()
    .sort("len", descending=True)
    .head(25)
)

stop_name,len
str,u32
"""Zürich HB""",27
"""Zürich Flughafen, Bahnhof""",18
"""Zürich Wiedikon, Bahnhof""",11
"""Zürich, Central""",11
"""Zürich Oerlikon""",10
…,…
"""Zürich, Luchswiesen""",7
"""Zürich, Klusplatz""",7
"""Zürich Enge, Bahnhof""",7


In [12]:
(
    zurich_stops
    .select("stop_name")
    .unique()
    .sort("stop_name")
)

stop_name
str
"""Zürich Affoltern"""
"""Zürich Affoltern, Bahnhof"""
"""Zürich Altstetten"""
"""Zürich Altstetten, Bahnhof"""
"""Zürich Altstetten, Bahnhof N"""
…
"""Zürich, Zwinglihaus"""
"""Zürich, Zypressenstrasse"""
"""Zürich, Zürichbergstrasse"""


### Observation

The naming convention captures:

- Zürich HB
- Zürich Flughafen
- Zürich Oerlikon
- Zürich Hardbrücke
- Zürich Wiedikon

while excluding most non-Zurich municipalities.

This produced a stable operational definition without requiring GIS boundaries or fare-zone mappings.

In [13]:
stop_times = pl.scan_parquet(
    PARQUET_DIR / "stop_times.parquet"
)

zurich_trip_ids = (
    stop_times
    .join(
        zurich_stops.lazy()
        .select("stop_id"),
        on="stop_id",
        how="semi",
    )
    .select("trip_id")
    .unique()
    .collect()
)

zurich_trip_ids.height

171622

In [15]:
total_trips = (
    pl.scan_parquet(
        PARQUET_DIR / "trips.parquet"
    )
    .select(
        pl.col("trip_id").n_unique()
    )
    .collect()
    .item()
)

pl.DataFrame(
    {
        "metric": ["Total Trips", "Zurich Trips"],
        "value": [total_trips, zurich_trip_ids.height],
    }
)

metric,value
str,i64
"""Total Trips""",1584909
"""Zurich Trips""",171622


In [16]:
trips = pl.scan_parquet(
    PARQUET_DIR / "trips.parquet"
)

zurich_routes = (
    trips
    .join(
        zurich_trip_ids.lazy(),
        on="trip_id",
        how="semi",
    )
    .select("route_id")
    .unique()
    .collect()
)

zurich_routes.height

261

In [17]:
pl.DataFrame(
    {
        "Metric": [
            "Stops",
            "Trips",
            "Routes",
        ],
        "Swiss Feed": [
            102796,
            1584909,
            5076,
        ],
        "Zurich Subset": [
            2020,
            171622,
            261,
        ],
    }
)

Metric,Swiss Feed,Zurich Subset
str,i64,i64
"""Stops""",102796,2020
"""Trips""",1584909,171622
"""Routes""",5076,261


## Conclusion

Final Zurich subset:

- Stops: 2,020
- Trips: 171,622
- Routes: 261

The subset retains approximately:

- 2% of Swiss stops
- 11% of Swiss trips
- 5% of Swiss routes

The resulting dataset forms the foundation for the Transit Intelligence Platform's Zurich-focused operational graph.

The production implementation persists these outputs as:

- zurich_stops.parquet
- zurich_trip_ids.parquet
- zurich_trips.parquet
- zurich_routes.parquet

under:

data/processed/

## Internal vs Crossing Service Classification

In [18]:
zurich_stop_ids = (
    zurich_stops
    .select("stop_id")
    .unique()
)

In [19]:
stop_times = pl.scan_parquet(
    PARQUET_DIR / "stop_times.parquet"
)

trip_stop_counts = (
    stop_times
    .group_by("trip_id")
    .agg(
        pl.len().alias("total_stops")
    )
)

In [20]:
zurich_stop_counts = (
    stop_times
    .join(
        zurich_stop_ids.lazy(),
        on="stop_id",
        how="semi"
    )
    .group_by("trip_id")
    .agg(
        pl.len().alias("zurich_stops")
    )
)

In [21]:
trip_classification = (
    trip_stop_counts
    .join(
        zurich_stop_counts,
        on="trip_id",
        how="inner"
    )
    .with_columns(
        pl.when(
            pl.col("total_stops")
            == pl.col("zurich_stops")
        )
        .then(pl.lit("internal"))
        .otherwise(pl.lit("crossing"))
        .alias("trip_type")
    )
    .collect()
)

In [23]:
(
    trip_classification
    .group_by("trip_type")
    .len()
)

trip_type,len
str,u32
"""crossing""",97664
"""internal""",73958


In [25]:
PROCESSED_DIR = PROJECT_ROOT / "processed"

In [26]:
zurich_trips = pl.read_parquet(
    PROCESSED_DIR / "trips" / "zurich_trips.parquet"
)

In [28]:
route_classification = (
    zurich_trips
    .join(
        trip_classification,
        on="trip_id"
    )
    .group_by("route_id")
    .agg(
        pl.col("trip_type")
        .unique()
    )
)

In [30]:
route_classification.head(20)

route_id,trip_type
str,list[str]
"""91-V-Y-j26-1""","[""crossing""]"
"""91-9B-Y-j26-1""","[""internal"", ""crossing""]"
"""92-492-D-j26-1""","[""crossing""]"
"""91-7Q-Y-j26-1""","[""crossing""]"
"""91-3-S-j26-1""","[""crossing""]"
…,…
"""92-N71-j26-1""","[""crossing""]"
"""92-N2-E-j26-1""","[""internal""]"
"""92-40-C-j26-1""","[""internal""]"


In [31]:
route_classification = (
    route_classification
    .with_columns(
        pl.when(
            pl.col("trip_type").list.len() == 1
        )
        .then(
            pl.col("trip_type").list.first()
        )
        .otherwise(
            pl.lit("mixed")
        )
        .alias("route_type")
    )
)

In [32]:
(
    route_classification
    .group_by("route_type")
    .len()
)

route_type,len
str,u32
"""crossing""",171
"""internal""",59
"""mixed""",31


In [33]:
(
    route_classification
    .group_by("route_type")
    .len()
    .with_columns(
        (
            pl.col("len")
            / pl.col("len").sum()
            * 100
        ).round(2)
        .alias("pct")
    )
)

route_type,len,pct
str,u32,f64
"""crossing""",171,65.52
"""mixed""",31,11.88
"""internal""",59,22.61


In [34]:
routes = pl.read_parquet(
    PARQUET_DIR / "routes.parquet"
)

route_details = (
    route_classification
    .join(
        routes,
        on = "route_id"
    )
)

In [35]:
route_details.filter(
    pl.col("route_type") == "mixed"
).select(
    "route_short_name",
    "route_long_name",
    "route_type"
).head(50)

route_short_name,route_long_name,route_type
str,str,str
"""S10""","""""","""mixed"""
"""10""","""""","""mixed"""
"""EXT""","""""","""mixed"""
"""S24""","""""","""mixed"""
"""2""","""""","""mixed"""
…,…,…
"""N18""","""""","""mixed"""
"""N4""","""""","""mixed"""
"""N7""","""""","""mixed"""
